# Predicción sobre test
Carga `modelo_final.joblib` y genera `submission.csv`.
Debe contener **solo** el código necesario para replicar la submission.

In [1]:
import pandas as pd
import joblib
from pathlib import Path
from features import aplanar_anidadas

## 1. Carga de datos y modelo

In [2]:
test = pd.read_json("data/test_data.jsonlines", lines=True)
test = aplanar_anidadas(test)

modelos_ajustados = joblib.load("modelos_ajustados.joblib")

## 2. Predicción y submission
CSV con columnas `ID` y `condition` (`new`/`used`), 30.000 filas, métrica de evaluación ACC.

**Probando:** `ID` como número de fila (1 a 30.000, orden de `test_data.jsonlines`) — el intento anterior con el `id` real de la publicación (`MLA...`) dio error de matching contra la solución de Kaggle. Si esto también falla, hay que revisar de nuevo.

In [3]:
nombres_archivo = {
    "Regresión logística": "logreg",
    "LDA": "lda",
    "QDA": "qda",
    "Naive Bayes (Gaussiano)": "nb",
    "kNN": "knn",
    "SVM lineal": "svm",
    "Árbol de decisión": "arbol",
    "Random Forest": "rf",
    "Extra Trees": "extra_trees",
    "AdaBoost": "adaboost",
    "Gradient Boosting": "histgb",
    "Voting (soft)": "voting_soft",
    "Voting (hard)": "voting_hard",
    "Stacking": "stacking",
}

Path("submission").mkdir(exist_ok=True)

for nombre, pipeline in modelos_ajustados.items():
    pred = pipeline.predict(test)
    condicion = pd.Series(pred).map({1: "new", 0: "used"})
    archivo = f"submission/submission_{nombres_archivo[nombre]}.csv"
    pd.DataFrame({"ID": range(1,len(test)+1), "condition": condicion}).to_csv(archivo, index=False)
    print(nombre, "->", archivo)

kNN -> submission/submission_knn.csv
SVM lineal -> submission/submission_svm.csv
Árbol de decisión -> submission/submission_arbol.csv
Random Forest -> submission/submission_rf.csv
Extra Trees -> submission/submission_extra_trees.csv
AdaBoost -> submission/submission_adaboost.csv
Gradient Boosting -> submission/submission_histgb.csv
Voting (soft) -> submission/submission_voting_soft.csv
Voting (hard) -> submission/submission_voting_hard.csv
Stacking -> submission/submission_stacking.csv
